In [2]:
# ------------------------- 1. IMPORTS -------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
# ------------------------- 2. LOAD DATASET -------------------------
print("="*100)
print("STEP 1: LOAD DATASET")
print("="*100)

file_path = 'DataCoSupplyChainDataset.csv'  # Update path if needed
df_raw = pd.read_csv(file_path, encoding='latin1')

print(f"✓ Dataset loaded successfully.")
print(f"  - Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"  - Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nFirst 3 rows preview:")
print(df_raw.head(3))

STEP 1: LOAD DATASET
✓ Dataset loaded successfully.
  - Shape: 180519 rows, 53 columns
  - Memory usage: 128.28 MB

First 3 rows preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  Benefit per order  Sales per customer   Delivery Status  Late_delivery_risk  Category Id   Category Name Customer City Customer Country Customer Email Customer Fname  Customer Id Customer Lname Customer Password Customer Segment Customer State           Customer Street  Customer Zipcode  Department Id Department Name  Latitude  Longitude        Market Order City Order Country  Order Customer Id order date (DateOrders)  Order Id  Order Item Cardprod Id  Order Item Discount  Order Item Discount Rate  Order Item Id  Order Item Product Price  Order Item Profit Ratio  Order Item Quantity  Sales  Order Item Total  Order Profit Per Order    Order Region      Order State Order Status  Order Zipcode  Product Card Id  Product Category Id  Product Description                                 P

In [4]:
# ------------------------- 3. INITIAL INSPECTION -------------------------
print("\n" + "="*100)
print("STEP 2: INITIAL INSPECTION")
print("="*100)

# Basic info
print("\n--- Data Types & Non-Null Counts ---")
df_raw.info()

print("\n--- Descriptive Statistics (Numerical) ---")
display(df_raw.describe().round(2))

print("\n--- Descriptive Statistics (Categorical) ---")
categorical_cols = df_raw.select_dtypes(include=['object']).columns
print(f"Number of categorical columns: {len(categorical_cols)}")
print("Sample categorical columns and unique counts:")
for col in categorical_cols[:5]:
    print(f"  - {col}: {df_raw[col].nunique()} unique values")


STEP 2: INITIAL INSPECTION

--- Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 18051

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Id,Customer Id,Customer Zipcode,Department Id,Latitude,Longitude,Order Customer Id,Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Price,Product Status
count,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180516.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,180519.00,24840.00,180519.00,180519.00,0.00,180519.00,180519.00
mean,3.50,2.93,21.97,183.11,0.55,31.85,6691.38,35921.13,5.44,29.72,-84.92,6691.38,36221.89,692.51,20.66,0.10,90260.00,141.23,0.12,2.13,203.77,183.11,21.97,55426.13,692.51,31.85,NaN,141.23,0.00
std,1.62,1.37,104.43,120.04,0.50,15.64,4162.92,37542.46,1.63,9.81,21.43,4162.92,21045.38,336.45,21.80,0.07,52111.49,139.73,0.47,1.45,132.27,120.04,104.43,31919.28,336.45,15.64,NaN,139.73,0.00
min,0.00,0.00,-4274.98,7.49,0.00,2.00,1.00,603.00,2.00,-33.94,-158.03,1.00,1.00,19.00,0.00,0.00,1.00,9.99,-2.75,1.00,9.99,7.49,-4274.98,1040.00,19.00,2.00,NaN,9.99,0.00
25%,2.00,2.00,7.00,104.38,0.00,18.00,3258.50,725.00,4.00,18.27,-98.45,3258.50,18057.00,403.00,5.40,0.04,45130.50,50.00,0.08,1.00,119.98,104.38,7.00,23464.00,403.00,18.00,NaN,50.00,0.00
50%,3.00,4.00,31.52,163.99,1.00,29.00,6457.00,19380.00,5.00,33.14,-76.85,6457.00,36140.00,627.00,14.00,0.10,90260.00,59.99,0.27,1.00,199.92,163.99,31.52,59405.00,627.00,29.00,NaN,59.99,0.00
75%,5.00,4.00,64.80,247.40,1.00,45.00,9779.00,78207.00,7.00,39.28,-66.37,9779.00,54144.00,1004.00,29.99,0.16,135389.50,199.99,0.36,3.00,299.95,247.40,64.80,90008.00,1004.00,45.00,NaN,199.99,0.00
max,6.00,4.00,911.80,1939.99,1.00,76.00,20757.00,99205.00,12.00,48.78,115.26,20757.00,77204.00,1363.00,500.00,0.25,180519.00,1999.99,0.50,5.00,1999.99,1939.99,911.80,99301.00,1363.00,76.00,NaN,1999.99,0.00



--- Descriptive Statistics (Categorical) ---
Number of categorical columns: 24
Sample categorical columns and unique counts:
  - Type: 4 unique values
  - Delivery Status: 4 unique values
  - Category Name: 50 unique values
  - Customer City: 563 unique values
  - Customer Country: 2 unique values


In [5]:
# ------------------------- 4. DATA QUALITY AUDIT -------------------------
print("\n" + "="*100)
print("STEP 3: DATA QUALITY AUDIT")
print("="*100)

# Missing values summary
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)
print("--- Missing Values ---")
if missing_df.empty:
    print("No missing values found.")
else:
    print(missing_df)

# Duplicate rows
duplicate_rows = df_raw.duplicated().sum()
print(f"\n--- Duplicate Rows: {duplicate_rows} ({duplicate_rows/len(df_raw)*100:.2f}%)")

# Data type issues (detect columns stored as object but should be numeric)
print("\n--- Potential Data Type Issues ---")
object_cols = df_raw.select_dtypes(include=['object']).columns
for col in object_cols:
    try:
        pd.to_numeric(df_raw[col])
        print(f"  - {col}: could be numeric (stored as object)")
    except:
        pass

# Unique value ratios for high-cardinality columns
print("\n--- High Cardinality Columns ---")
for col in categorical_cols:
    unique_ratio = df_raw[col].nunique() / len(df_raw)
    if unique_ratio > 0.05:  # more than 5% unique values
        print(f"  - {col}: {df_raw[col].nunique()} unique values ({unique_ratio:.1%})")


STEP 3: DATA QUALITY AUDIT
--- Missing Values ---
                     Missing Count  Percentage
Product Description         180519      100.00
Order Zipcode               155679       86.24
Customer Lname                   8        0.00
Customer Zipcode                 3        0.00

--- Duplicate Rows: 0 (0.00%)

--- Potential Data Type Issues ---

--- High Cardinality Columns ---
  - order date (DateOrders): 65752 unique values (36.4%)
  - shipping date (DateOrders): 63701 unique values (35.3%)


In [6]:
# ------------------------- 5. STANDARDIZE COLUMN NAMES -------------------------
print("\n" + "="*100)
print("STEP 4: STANDARDIZE COLUMN NAMES")
print("="*100)

def standardize_column_names(df):
    """Convert column names to lowercase, replace spaces with underscores, remove special chars."""
    new_cols = []
    for col in df.columns:
        col = col.lower().strip()
        col = col.replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')
        col = col.replace('%', 'percent')
        new_cols.append(col)
    return new_cols

df = df_raw.copy()
original_cols = df.columns.tolist()
df.columns = standardize_column_names(df)
print("Column names standardized:")
print(f"  - Original: {original_cols[:3]} ...")
print(f"  - New:      {df.columns[:3]} ...")
print(f"  - Total columns renamed: {len(original_cols)}")


STEP 4: STANDARDIZE COLUMN NAMES
Column names standardized:
  - Original: ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)'] ...
  - New:      Index(['type', 'days_for_shipping_real', 'days_for_shipment_scheduled'], dtype='str') ...
  - Total columns renamed: 53


In [7]:
# ------------------------- 6. REMOVE PII & IRRELEVANT COLUMNS -------------------------
print("\n" + "="*100)
print("STEP 5: REMOVE PII & IRRELEVANT COLUMNS")
print("="*100)

pii_irrelevant = [
    'customer_email', 'customer_password', 'customer_fname', 'customer_lname',
    'customer_id', 'order_customer_id', 'product_image', 'product_description',
    'order_id', 'order_item_id', 'product_card_id', 'category_id', 'department_id',
    'customer_zipcode', 'order_zipcode', 'latitude', 'longitude', 'order_item_cardprod_id'
]
cols_to_drop = [col for col in pii_irrelevant if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"✓ Removed {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"  - New shape: {df.shape}")


STEP 5: REMOVE PII & IRRELEVANT COLUMNS
✓ Removed 18 columns: ['customer_email', 'customer_password', 'customer_fname', 'customer_lname', 'customer_id', 'order_customer_id', 'product_image', 'product_description', 'order_id', 'order_item_id', 'product_card_id', 'category_id', 'department_id', 'customer_zipcode', 'order_zipcode', 'latitude', 'longitude', 'order_item_cardprod_id']
  - New shape: (180519, 35)


In [8]:
# ------------------------- 7. MISSING VALUE TREATMENT -------------------------
print("\n" + "="*100)
print("STEP 6: MISSING VALUE TREATMENT")
print("="*100)

# Check missing again after column removal
missing_new = df.isnull().sum()
missing_new_pct = (missing_new / len(df)) * 100
missing_summary = pd.DataFrame({'Missing': missing_new, 'Percentage': missing_new_pct})
missing_summary = missing_summary[missing_summary['Missing'] > 0].sort_values('Percentage', ascending=False)
print("Missing values remaining:")
print(missing_summary)

# Strategy:
# - Drop rows where target 'benefit_per_order' is missing (if any)
# - Fill categorical with 'Unknown'
# - Fill numeric with median

if df['benefit_per_order'].isnull().sum() > 0:
    df.dropna(subset=['benefit_per_order'], inplace=True)
    print(f"  - Dropped rows with missing target. New shape: {df.shape}")

# Categorical fill
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('Unknown', inplace=True)
        print(f"  - Filled missing in '{col}' with 'Unknown'")

# Numeric fill (median)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  - Filled missing in '{col}' with median ({median_val:.2f})")

print("\n✓ All missing values handled.")
print(f"  - Final shape: {df.shape}")
print(f"  - Total missing count: {df.isnull().sum().sum()}")



STEP 6: MISSING VALUE TREATMENT
Missing values remaining:
Empty DataFrame
Columns: [Missing, Percentage]
Index: []

✓ All missing values handled.
  - Final shape: (180519, 35)
  - Total missing count: 0


In [9]:
# ------------------------- 8. DUPLICATE DETECTION & REMOVAL -------------------------
print("\n" + "="*100)
print("STEP 7: DUPLICATE DETECTION & REMOVAL")
print("="*100)

duplicates = df.duplicated().sum()
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"✓ Removed {duplicates} duplicate rows.")
else:
    print("✓ No duplicate rows found.")
print(f"  - Shape after deduplication: {df.shape}")


STEP 7: DUPLICATE DETECTION & REMOVAL
✓ No duplicate rows found.
  - Shape after deduplication: (180519, 35)


In [10]:
# ------------------------- 9. DATA TYPE OPTIMIZATION -------------------------
print("\n" + "="*100)
print("STEP 8: DATA TYPE OPTIMIZATION")
print("="*100)

# Convert object columns that are actually numeric
for col in df.select_dtypes(include=['object']).columns:
    try:
        df[col] = pd.to_numeric(df[col])
        print(f"  - Converted '{col}' from object to numeric.")
    except:
        pass

# Convert boolean-like columns to int
bool_like = ['late_delivery_risk']
for col in bool_like:
    if col in df.columns:
        df[col] = df[col].astype(int)
        print(f"  - Converted '{col}' to int (binary).")

# Downcast integers to save memory
int_cols = df.select_dtypes(include=['int64']).columns
for col in int_cols:
    df[col] = pd.to_numeric(df[col], downcast='integer')
    
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast='float')
    
print("✓ Data types optimized (downcast to smallest possible).")
print(df.dtypes.value_counts())


STEP 8: DATA TYPE OPTIMIZATION
  - Converted 'late_delivery_risk' to int (binary).
✓ Data types optimized (downcast to smallest possible).
str        19
float32    10
int8        6
Name: count, dtype: int64


In [11]:
# ------------------------- 10. DATE PROCESSING -------------------------
print("\n" + "="*100)
print("STEP 9: DATE PROCESSING")
print("="*100)

date_cols = ['order_date_(dateorders)', 'shipping_date_(dateorders)']
for col in date_cols:
    if col in df.columns:
        # Convert to datetime
        df[col] = pd.to_datetime(df[col], errors='coerce')
        # Extract datetime features
        df[f'{col}_year'] = df[col].dt.year
        df[f'{col}_month'] = df[col].dt.month
        df[f'{col}_dayofweek'] = df[col].dt.dayofweek
        df[f'{col}_quarter'] = df[col].dt.quarter
        # Drop original if not needed (optional: keep for time series)
        # df.drop(col, axis=1, inplace=True)
        print(f"  - Processed '{col}': extracted year, month, dayofweek, quarter.")
print("✓ Date features created.")


STEP 9: DATE PROCESSING
✓ Date features created.


In [12]:
# ------------------------- 11. DATA VALIDATION CHECKS -------------------------
print("\n" + "="*100)
print("STEP 10: DATA VALIDATION CHECKS")
print("="*100)

# Check for negative values in columns that should be non-negative
non_negative_cols = ['days_for_shipping_(real)', 'days_for_shipment_(scheduled)', 'order_item_quantity', 'sales']
for col in non_negative_cols:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  - Warning: '{col}' has {neg_count} negative values.")
            # Option: clip to 0
            df.loc[df[col] < 0, col] = 0
        else:
            print(f"  - '{col}' has no negative values. ✓")

# Check if 'benefit_per_order' (target) is within realistic range
print(f"\nTarget variable 'benefit_per_order' range: [{df['benefit_per_order'].min():.2f}, {df['benefit_per_order'].max():.2f}]")
print(f"  - Negative profits exist: {(df['benefit_per_order'] < 0).sum()} rows ({(df['benefit_per_order'] < 0).mean()*100:.1f}%)")

# Logical consistency: shipping real days should not be less than scheduled days?
if 'days_for_shipping_(real)' in df and 'days_for_shipment_(scheduled)' in df:
    invalid_shipping = (df['days_for_shipping_(real)'] < df['days_for_shipment_(scheduled)']).sum()
    print(f"  - Rows where real shipping < scheduled shipping: {invalid_shipping} (possible early delivery)")


STEP 10: DATA VALIDATION CHECKS
  - 'order_item_quantity' has no negative values. ✓
  - 'sales' has no negative values. ✓

Target variable 'benefit_per_order' range: [-4274.98, 911.80]
  - Negative profits exist: 33784 rows (18.7%)


In [13]:
# ------------------------- 12. FEATURE ENGINEERING -------------------------
print("\n" + "="*100)
print("STEP 11: FEATURE ENGINEERING")
print("="*100)

# Shipping delay (positive = delayed, negative = early)
if 'days_for_shipping_(real)' in df and 'days_for_shipment_(scheduled)' in df:
    df['shipping_delay'] = df['days_for_shipping_(real)'] - df['days_for_shipment_(scheduled)']
    print("  - Created 'shipping_delay' (real - scheduled).")

# Profit margin ratio (Benefit / Sales)
if 'benefit_per_order' in df and 'sales' in df:
    df['profit_margin_ratio'] = df['benefit_per_order'] / (df['sales'] + 1e-6)
    print("  - Created 'profit_margin_ratio' (benefit/sales).")

# Product name length (proxy for description detail)
if 'product_name' in df:
    df['product_name_length'] = df['product_name'].str.len()
    print("  - Created 'product_name_length'.")

# Total item value (unit price * quantity) – sanity check with sales
if 'order_item_product_price' in df and 'order_item_quantity' in df:
    df['calculated_sales'] = df['order_item_product_price'] * df['order_item_quantity']
    # Optional: validate against sales column
    df['sales_diff'] = abs(df['sales'] - df['calculated_sales'])
    print("  - Created 'calculated_sales' for validation.")
    # Drop after validation
    df.drop(['calculated_sales', 'sales_diff'], axis=1, inplace=True)

# Weekend flag (if order date exists)
if 'order_date_(dateorders)_dayofweek' in df:
    df['is_weekend'] = (df['order_date_(dateorders)_dayofweek'] >= 5).astype(int)
    print("  - Created 'is_weekend' flag.")

print("✓ Feature engineering completed.")



STEP 11: FEATURE ENGINEERING
  - Created 'profit_margin_ratio' (benefit/sales).
  - Created 'product_name_length'.
  - Created 'calculated_sales' for validation.
✓ Feature engineering completed.


In [14]:
# ------------------------- 13. MEMORY OPTIMIZATION -------------------------
print("\n" + "="*100)
print("STEP 12: MEMORY OPTIMIZATION")
print("="*100)

# Downcast again after new columns
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

# Convert object columns with low cardinality to category
for col in df.select_dtypes(include=['object']).columns:
    if df[col].nunique() / len(df) < 0.05:  # less than 5% unique
        df[col] = df[col].astype('category')
        print(f"  - Converted '{col}' to category (memory efficient).")

final_memory = df.memory_usage(deep=True).sum() / 1024**2
print(f"\n✓ Memory usage reduced to {final_memory:.2f} MB (from original ~{df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB).")


STEP 12: MEMORY OPTIMIZATION
  - Converted 'type' to category (memory efficient).
  - Converted 'delivery_status' to category (memory efficient).
  - Converted 'category_name' to category (memory efficient).
  - Converted 'customer_city' to category (memory efficient).
  - Converted 'customer_country' to category (memory efficient).
  - Converted 'customer_segment' to category (memory efficient).
  - Converted 'customer_state' to category (memory efficient).
  - Converted 'customer_street' to category (memory efficient).
  - Converted 'department_name' to category (memory efficient).
  - Converted 'market' to category (memory efficient).
  - Converted 'order_city' to category (memory efficient).
  - Converted 'order_country' to category (memory efficient).
  - Converted 'order_region' to category (memory efficient).
  - Converted 'order_state' to category (memory efficient).
  - Converted 'order_status' to category (memory efficient).
  - Converted 'product_name' to category (memory e

In [15]:
# ------------------------- 14. CREATE VISUALIZATION DATASET -------------------------
print("\n" + "="*100)
print("STEP 13: CREATE VISUALIZATION DATASET")
print("="*100)

# Keep a copy with original categorical values (not encoded) for plotting
df_viz = df.copy()
print("✓ Visualization dataset created (categorical columns retained as original).")
print(f"  - Shape: {df_viz.shape}")



STEP 13: CREATE VISUALIZATION DATASET
✓ Visualization dataset created (categorical columns retained as original).
  - Shape: (180519, 37)


In [16]:
# ------------------------- 15. CREATE ML DATASET (ENCODED & SCALED) -------------------------
print("\n" + "="*100)
print("STEP 14: CREATE ML DATASET (Profit Prediction)")
print("="*100)

# Define target
target = 'benefit_per_order'
X = df.drop(columns=[target])
y = df[target]

# Identify categorical columns (object or category)
cat_cols_ml = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns for encoding: {cat_cols_ml}")

# One-hot encoding (drop first to avoid multicollinearity)
X_encoded = pd.get_dummies(X, columns=cat_cols_ml, drop_first=True)
print(f"Shape after one-hot encoding: {X_encoded.shape}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale numeric features (all remaining columns after encoding are numeric)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[X_train.columns] = scaler.fit_transform(X_train)
X_test_scaled[X_test.columns] = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler.")

# Save processed ML dataset (optional)
df_ml = pd.concat([X_train_scaled, y_train], axis=1)
df_ml.to_csv('DataCo_ml_ready_train.csv', index=False)
print("✓ ML training dataset saved as 'DataCo_ml_ready_train.csv'.")


STEP 14: CREATE ML DATASET (Profit Prediction)
Categorical columns for encoding: ['type', 'delivery_status', 'category_name', 'customer_city', 'customer_country', 'customer_segment', 'customer_state', 'customer_street', 'department_name', 'market', 'order_city', 'order_country', 'order_date_dateorders', 'order_region', 'order_state', 'order_status', 'product_name', 'shipping_date_dateorders', 'shipping_mode']


MemoryError: Unable to allocate 11.1 GiB for an array with shape (180519, 65752) and data type bool

In [ ]:
# ------------------------- 16. FINAL CLEANING SUMMARY -------------------------
print("\n" + "=" * 100)
print("STEP 15: FINAL CLEANING SUMMARY")
print("=" * 100)

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║                           DATA CLEANING REPORT                                 ║
╠════════════════════════════════════════════════════════════════════════════════╣
║ Original dataset:         {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns
║ Final cleaned dataset:    {df.shape[0]:,} rows, {df.shape[1]} columns
║ Rows removed (duplicates): {duplicates}
║ Columns removed (PII/Irr): {len(cols_to_drop)}
║ Missing values handled:    {missing_df["Missing Count"].sum() if not missing_df.empty else 0}
║ New features engineered:   5+ (shipping_delay, profit_margin_ratio, etc.)
║ Memory usage reduction:    {final_memory:.2f} MB (from {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB)
║ ML dataset shape:          {X_encoded.shape[0]:,} rows, {X_encoded.shape[1]} features
║ Target variable:           {target}
║ Ready for:                 ✓ Exploratory Data Analysis
║                            ✓ Visualization (Poster/Report)
║                            ✓ Predictive Modeling (Profit Prediction)
╚════════════════════════════════════════════════════════════════════════════════╝
""")

print("=" * 100)
print("CLEANING PIPELINE COMPLETED SUCCESSFULLY.")
print("=" * 100)


STEP 15: FINAL CLEANING SUMMARY

╔════════════════════════════════════════════════════════════════════════════════╗
║                           DATA CLEANING REPORT                                 ║
╠════════════════════════════════════════════════════════════════════════════════╣
║ Original dataset:         180,519 rows, 53 columns
║ Final cleaned dataset:    180,519 rows, 37 columns
║ Rows removed (duplicates): 0
║ Columns removed (PII/Irr): 18
║ Missing values handled:    336209
║ New features engineered:   5+ (shipping_delay, profit_margin_ratio, etc.)
║ Memory usage reduction:    20.61 MB (from 128.28 MB)
║ ML dataset shape:          180,519 rows, 142601 features
║ Target variable:           benefit_per_order
║ Ready for:                 ✓ Exploratory Data Analysis
║                            ✓ Visualization (Poster/Report)
║                            ✓ Predictive Modeling (Profit Prediction)
╚════════════════════════════════════════════════════════════════════════════════╝

CL